# Geothermal AI Workflow  automated LAS-to-petrophysics pipeline

Given a LAS file, a deviation survey, and (optionally) a lithostratigraphy table, this pipeline produces a clean reservoir-interval petrophysical summary — TVD-corrected, anomaly-filtered, with Vshale, effective porosity, and NTG.

## Pipeline steps

1. **Ingest** LAS file with `lasio`
2. **QC** — Isolation Forest anomaly detection per curve (the ML element)
3. **Convert** MD to TVD using the deviation survey
4. **Locate** the Slochteren reservoir interval from lithostratigraphy
5. **Compute** petrophysics: density-porosity, Vshale (Larionov), effective porosity
6. **Summarize** — one-row reservoir characterization

## Why "AI-assisted"?

The Isolation Forest QC step flags the ~1% most statistically isolated points per curve (fixed `contamination=0.01`, `random_state=42`) — no per-curve physical thresholds. This is what caught a real issue in our data: the JUT-01 LAS file contains RHOB values up to 7.5 million g/cc (sensor or transcription error). Note that hard physical limits (e.g. the porosity clip) are enforced separately in `compute_petrophysics`, so the fixed 1% trim is a statistical outlier flag, not the physical-validity gate.

In [ ]:
import sys
sys.path.insert(0, '../pipeline')
from pipeline import run_pipeline, ingest_las, qc_curves, md_to_tvd_interpolator, add_tvd_columns, compute_petrophysics, find_reservoir_interval

import pandas as pd
import numpy as np

DATA_RAW = '../../01_data_raw'
DATA_PROCESSED = '../../02_data_processed'

print('Libraries loaded')

## Step 1  Demonstrate the LAS ingest

In [ ]:
df_blt, las_blt = ingest_las(f'{DATA_RAW}/well_logs_las/blt01.las')
print(f"BLT-01: {len(df_blt)} rows")
print(f"Curves available: {[c for c in df_blt.columns if c != 'MD']}")
df_blt.head()

## Step 2 Anomaly detection (Isolation Forest)

Demonstrated on JUT-01 RHOB, which contains values up to 7.5 million g/cc.

In [ ]:
df_jut, _ = ingest_las(f'{DATA_RAW}/well_logs_las/jut01.las')
print(f"Before QC: JUT-01 RHOB range = [{df_jut['RHOB'].min():.3f}, {df_jut['RHOB'].max():,.1f}]")

df_jut_clean, qc_report = qc_curves(df_jut)
print(f"After QC:  JUT-01 RHOB range = [{df_jut_clean['RHOB'].min():.3f}, {df_jut_clean['RHOB'].max():.3f}]")
print("\nAnomaly report:")
qc_report

## Step 3  Run the full pipeline on all four wells

In [ ]:
wells = ['BLT-01', 'EVD-01', 'JUT-01', 'PKP-01']
summaries = []
qc_reports = {}

for w in wells:
    slug = w.lower().replace('-', '')
    df, summary, qc = run_pipeline(
        well_id=w,
        las_path=f'{DATA_RAW}/well_logs_las/{slug}.las',
        survey_path=f'{DATA_PROCESSED}/tvd_conversions/{slug}_deviation_survey.csv',
        litho_path=f'{DATA_RAW}/lithostratigraphy/lithostratigraphic_data.xlsx',
        litho_sheet=w
    )
    summaries.append(summary)
    qc_reports[w] = qc

summary_df = pd.DataFrame(summaries)
summary_df[['well_id', 'top_tvd_m', 'base_tvd_m', 'thickness_tvd_m', 
            'phid_mean', 'phie_mean', 'ntg', 'n_log_samples']]

## Step 4  Inspect anomaly reports per well

In [ ]:
for w, qc in qc_reports.items():
    if len(qc) == 0:
        print(f"{w}: no anomalies")
    else:
        print(f"\n{w}:")
        print(qc.to_string(index=False))

## Headline results

| Well | Φ_eff | NTG | Reservoir quality |
|---|---|---|---|
| BLT-01 | 11.4% | 0.93 | Good — primary candidate |
| JUT-01 | 10.9% | 0.86 | Good — secondary candidate |
| EVD-01 | 8.0% | 0.82 | Tight |
| PKP-01 | 1.1% | 0.03 | Dead reservoir at this location |

**Critical finding**: PKP-01 ThermoGIS reports 9% porosity, but actual LAS-derived effective porosity is 1.1%. Regional models can mislead — well-specific log analysis catches this.

#

Manually, per well: ~2 hours of cleaning, MD→TVD, petrophysics, write-up.
This pipeline: ~30 seconds per well, fully reproducible audit trail.

Scaled to 50 wells (basin screening): ~100 hours of analyst time saved per assessment cycle.

